# 03 - Convert Hive and HDFS datasets to Polaris Iceberg tables

Read orders through Hive metadata, read products directly from HDFS CSV, and read customers directly from HDFS JSON. Rewrite all three into Iceberg tables registered in Polaris, with their data and metadata files stored in SeaweedFS S3.

Run notebook 02 and stop its Spark session first. This notebook does not connect to MySQL. The source files and Hive table remain available after conversion. This is a physical rewrite into S3; it does not register the original HDFS files as Iceberg files.

In [ ]:
import os
import re
import socket
from getpass import getpass

MYSQL_DB = "training"
TABLE_PREFIX = "lake16_"
LAB = "mysql_hdfs_2"
assert all(re.fullmatch(r"[a-z][a-z0-9_]*", value) for value in [MYSQL_DB, TABLE_PREFIX, LAB])
HDFS_URI = os.getenv("WORKSHOP_HDFS_URI", "hdfs://namenode:9000").rstrip("/")
assert HDFS_URI.startswith("hdfs://"), "Choose HDFS, not file:/// or S3."
HIVE_URI = os.getenv("HIVE_METASTORE_URI", "thrift://hive-metastore:9083")
HDFS_ROOT = f"{HDFS_URI}/workshop/16-mysql-hdfs-iceberg/{LAB}"
ORDERS_PATH = HDFS_ROOT + "/orders_parquet"
PRODUCTS_PATH = HDFS_ROOT + "/products_csv"
CUSTOMERS_PATH = HDFS_ROOT + "/customers_json"
HIVE_ORDERS = f"spark_catalog.{LAB}.orders_external_2"
LAKE_ROOT = f"s3://warehouse/iceberg/{LAB}"
print("HDFS root:", HDFS_ROOT)

## Start Spark with Hive, Polaris and SeaweedFS
The existing environment supplies catalog and S3 credentials. Each notebook contains all of its own setup code.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pathlib import Path
assert SparkSession.getActiveSession() is None, "Stop the previous Spark session and restart this kernel."
jars = Path(os.getenv("SPARK_HOME", "/opt/spark")) / "jars"
assert list(jars.glob("mysql-connector-j-*.jar")), "Use the Spark image with its MySQL JDBC driver."
builder = (SparkSession.builder.appName("mysql-hdfs-" + LAB)
    .master(os.getenv("SPARK_MASTER_URL", "spark://spark-master:7077"))
    .config("spark.driver.host", os.getenv("POD_IP") or socket.gethostbyname(socket.gethostname()))
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.cores.max", "1").config("spark.executor.cores", "1")
    .config("spark.executor.memory", "512m")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.redaction.regex", "(?i)secret|password|token|access.?key|credential")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
    .config("spark.hadoop.dfs.replication", "1"))

builder = (builder.config("spark.hadoop.hive.metastore.uris", HIVE_URI)
    .config("spark.sql.warehouse.dir", HDFS_ROOT + "/managed_warehouse")
    .enableHiveSupport())

required = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "POLARIS_CLIENT_ID", "POLARIS_CLIENT_SECRET"]
assert all(os.getenv(k) for k in required), "Use the existing credential environment wiring."
uri = os.getenv("POLARIS_URI", "http://polaris:8181/api/catalog").rstrip("/")
endpoint = os.getenv("S3_ENDPOINT", "http://seaweedfs:8333")
settings = {
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    "spark.sql.catalog.polaris": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.polaris.type": "rest",
    "spark.sql.catalog.polaris.uri": uri,
    "spark.sql.catalog.polaris.warehouse": "training",
    "spark.sql.catalog.polaris.rest.auth.type": "oauth2",
    "spark.sql.catalog.polaris.oauth2-server-uri": uri + "/v1/oauth/tokens",
    "spark.sql.catalog.polaris.scope": "PRINCIPAL_ROLE:ALL",
    "spark.sql.catalog.polaris.header.Polaris-Realm": "POLARIS",
    "spark.sql.catalog.polaris.credential": os.environ["POLARIS_CLIENT_ID"] + ":" + os.environ["POLARIS_CLIENT_SECRET"],
    "spark.sql.catalog.polaris.io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
    "spark.sql.catalog.polaris.s3.endpoint": endpoint,
    "spark.sql.catalog.polaris.s3.path-style-access": "true",
    "spark.sql.catalog.polaris.s3.access-key-id": os.environ["AWS_ACCESS_KEY_ID"],
    "spark.sql.catalog.polaris.s3.secret-access-key": os.environ["AWS_SECRET_ACCESS_KEY"],
    "spark.sql.catalog.polaris.client.region": "us-east-1",
    "spark.hadoop.fs.s3a.endpoint": endpoint,
    "spark.hadoop.fs.s3a.path.style.access": "true",
    "spark.hadoop.fs.s3a.connection.ssl.enabled": "false",
    "spark.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    "spark.hadoop.fs.s3a.access.key": os.environ["AWS_ACCESS_KEY_ID"],
    "spark.hadoop.fs.s3a.secret.key": os.environ["AWS_SECRET_ACCESS_KEY"],
}
for key, value in settings.items():
    builder = builder.config(key, value)

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version, "master:", spark.sparkContext.master)

In [ ]:
# Use the Hadoop client already loaded by Spark; no host CLI or local staging is needed.
j = spark.sparkContext._jvm
hadoop = spark.sparkContext._jsc.hadoopConfiguration()
hdfs = j.org.apache.hadoop.fs.FileSystem.get(j.java.net.URI(HDFS_URI), hadoop)
assert hdfs.getUri().toString().startswith("hdfs://")
print("HDFS:", hdfs.getUri().toString())
assert hdfs.exists(j.org.apache.hadoop.fs.Path("/")), "HDFS is not ready."

In [ ]:
PRODUCT_SCHEMA = "product_id BIGINT, product_name STRING, category STRING, list_price DECIMAL(12,2)"
CUSTOMER_SCHEMA = "customer_id BIGINT, customer_name STRING, country STRING, email STRING"
ORDER_COLUMNS = ["order_id", "customer_id", "product_id", "quantity", "unit_price", "country"]
PRODUCT_COLUMNS = ["product_id", "product_name", "category", "list_price"]
CUSTOMER_COLUMNS = ["customer_id", "customer_name", "country", "email"]

## Read all three HDFS sources
The Hive read resolves a table through the metastore. CSV and JSON reads resolve directory paths directly.

In [ ]:
assert spark.catalog.tableExists(HIVE_ORDERS), "Run notebook 02 first."
orders = spark.table(HIVE_ORDERS).select(*ORDER_COLUMNS)
products = (spark.read.schema(PRODUCT_SCHEMA).option("header", True).option("escape", '"')
    .option("nullValue", "\\N").option("mode", "FAILFAST").csv(PRODUCTS_PATH))
customers = spark.read.schema(CUSTOMER_SCHEMA).option("mode", "FAILFAST").json(CUSTOMERS_PATH)
assert orders.count() == 6 and products.count() == 4 and customers.count() == 4
assert products.filter("category IS NULL").count() == 1
assert customers.filter("email IS NULL").count() == 1
print("PASS: all HDFS sources available with expected types and nulls")

## Create the Polaris namespace and Iceberg tables

The Polaris catalog is `training`; `polaris` is its Spark alias. Each table receives its own location under the existing allowed `s3://warehouse/iceberg` root. Orders retain country partitioning. The other two small tables are unpartitioned.

The initial CTAS creates an empty Iceberg table with the source schema. `overwrite(lit(True))` replaces its full contents using an Iceberg commit. Rerunning produces the same current rows, with additional snapshots possible. Keep source writers and concurrent target writers stopped during this small full-snapshot example.

In [ ]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS polaris.{LAB} LOCATION '{LAKE_ROOT}'")
orders.createOrReplaceTempView("hdfs_orders")
products.createOrReplaceTempView("hdfs_products")
customers.createOrReplaceTempView("hdfs_customers")
spark.sql(f"""CREATE TABLE IF NOT EXISTS polaris.{LAB}.orders_2 USING iceberg
    PARTITIONED BY (country) LOCATION '{LAKE_ROOT}/orders_2'
    TBLPROPERTIES ('format-version'='2') AS SELECT * FROM hdfs_orders WHERE 1=0""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS polaris.{LAB}.products_2 USING iceberg
    LOCATION '{LAKE_ROOT}/products_2' TBLPROPERTIES ('format-version'='2')
    AS SELECT * FROM hdfs_products WHERE 1=0""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS polaris.{LAB}.customers_2 USING iceberg
    LOCATION '{LAKE_ROOT}/customers_2' TBLPROPERTIES ('format-version'='2')
    AS SELECT * FROM hdfs_customers WHERE 1=0""")
orders.writeTo(f"polaris.{LAB}.orders_2").overwrite(F.lit(True))
products.writeTo(f"polaris.{LAB}.products_2").overwrite(F.lit(True))
customers.writeTo(f"polaris.{LAB}.customers_2").overwrite(F.lit(True))
print("PASS: three Iceberg tables committed through Polaris")

## Reconcile rows and verify S3 file locations

Check both directions so missing rows, extra rows and duplicate changes are detected. Iceberg's `files` metadata table shows the physical Parquet objects. Every target file must use the SeaweedFS warehouse bucket, rather than HDFS.

In [ ]:
for name, source, columns in [("orders_2",orders,ORDER_COLUMNS), ("products_2",products,PRODUCT_COLUMNS), ("customers_2",customers,CUSTOMER_COLUMNS)]:
    target = spark.table(f"polaris.{LAB}.{name}").select(*columns)
    assert source.select(*columns).exceptAll(target).count() == 0
    assert target.exceptAll(source.select(*columns)).count() == 0
    files = spark.sql(f"SELECT file_path, record_count FROM polaris.{LAB}.{name}.files")
    paths = [r.file_path for r in files.collect()]
    assert paths and all(p.startswith(LAKE_ROOT + "/" + name + "/") for p in paths)
    files.show(truncate=False)
    spark.sql(f"SELECT snapshot_id, operation FROM polaris.{LAB}.{name}.snapshots").show(truncate=False)
print("PASS: all Iceberg rows match their HDFS sources and all target files are in S3")

## Query the converted data
Join the three Iceberg tables. Use the order selling price for revenue; product list prices are descriptive attributes.

In [ ]:
spark.sql(f"""SELECT c.country, count(*) AS orders, sum(o.quantity*o.unit_price) AS revenue
    FROM polaris.{LAB}.orders_2 o
    JOIN polaris.{LAB}.customers_2 c ON o.customer_id=c.customer_id
    JOIN polaris.{LAB}.products_2 p ON o.product_id=p.product_id
    GROUP BY c.country ORDER BY c.country""").show()
joined = spark.sql(f"""SELECT count(*) AS orders, sum(o.quantity*o.unit_price) AS revenue
    FROM polaris.{LAB}.orders_2 o
    JOIN polaris.{LAB}.customers_2 c ON o.customer_id=c.customer_id
    JOIN polaris.{LAB}.products_2 p ON o.product_id=p.product_id""").first()
assert joined.orders == 6 and joined.revenue == 190
print("PASS: six joined orders and revenue 190.00")

## Record repeat-run fingerprints
Run the workflow twice. Counts and fingerprints must match, although file names and Iceberg snapshot IDs can change.

In [ ]:
import hashlib
import json
results = {}
for name, frame in [("hdfs_orders",orders), ("hdfs_products",products), ("hdfs_customers",customers),
    ("iceberg_orders",spark.table(f"polaris.{LAB}.orders_2")),
    ("iceberg_products",spark.table(f"polaris.{LAB}.products_2")),
    ("iceberg_customers",spark.table(f"polaris.{LAB}.customers_2"))]:
    rows = sorted(frame.toJSON().collect())
    results[name] = {"rows":len(rows), "sha256":hashlib.sha256("\n".join(rows).encode()).hexdigest()}
print("PASS RESULT_FINGERPRINT " + json.dumps(results,sort_keys=True))

## Release Spark
Run this cell before starting the next notebook. Use a fresh kernel for the next notebook.

In [ ]:
spark.stop()